# VisionTrack - Day 3: KITTI Traffic Detector Model Fine-Tuning
This notebook trains a custom YOLO traffic detector on the KITTI dataset using a free Google Colab GPU (T4/V100).

**Target Classes (5)**:
- 0: `car`
- 1: `truck`
- 2: `bus`
- 3: `pedestrian`
- 4: `cyclist`

## 1. Verify GPU Acceleration
Ensure your Colab runtime is set to **GPU** (`Runtime -> Change runtime type -> T4 GPU`).

In [ ]:
!nvidia-smi

## 2. Install Ultralytics and Dependencies

In [ ]:
!pip install --upgrade ultralytics pyyaml

## 3. Dataset Setup
Upload your `kitti_yolo` dataset zip file (or mount Google Drive).
If uploading directly via Colab, run the cell below to extract it.

In [ ]:
import os
import zipfile

# If you uploaded kitti_yolo.zip, unzip it here:
if os.path.exists('kitti_yolo.zip'):
    with zipfile.ZipFile('kitti_yolo.zip', 'r') as zip_ref:
        zip_ref.extractall('data/processed/')
    print('Dataset extracted successfully.')
else:
    print('Upload kitti_yolo.zip or clone your repo.')

## 4. Define Dataset Configuration (`traffic_kitti.yaml`)

In [ ]:
yaml_content = """
path: data/processed/kitti_yolo
train: images/train
val: images/val
test: images/test

names:
  0: car
  1: truck
  2: bus
  3: pedestrian
  4: cyclist
"""

os.makedirs('configs', exist_ok=True)
with open('configs/traffic_kitti.yaml', 'w') as f:
    f.write(yaml_content.strip())
print('Config saved to configs/traffic_kitti.yaml')

## 5. Execute Model Fine-Tuning (30 Epochs)
Using the exact hyperparameters configured during Day 3:
- Backbone freezing: `freeze=10`
- Optimizer: `AdamW` (lr0=0.01, lrf=0.01)
- Data augmentation: `mosaic=1.0`, `mixup=0.1`, `close_mosaic=10`

In [ ]:
from ultralytics import YOLO

# Load pre-trained nano detector
model = YOLO('yolov8n.pt')

# Fine-tune on KITTI traffic dataset
results = model.train(
    data='configs/traffic_kitti.yaml',
    epochs=30,
    batch=16,
    imgsz=640,
    freeze=10,
    optimizer='AdamW',
    lr0=0.01,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    mosaic=1.0,
    mixup=0.1,
    close_mosaic=10,
    device=0,
    project='runs/train',
    name='traffic_kitti_v1'
)

## 6. Evaluate and Visualize Training Curves
Inspect loss decay (CIoU box loss, BCE classification loss, DFL loss) and mAP convergence.

In [ ]:
from IPython.display import Image, display

# Show training loss and mAP curves
results_img = 'runs/train/traffic_kitti_v1/results.png'
if os.path.exists(results_img):
    display(Image(results_img))

# Show confusion matrix
cm_img = 'runs/train/traffic_kitti_v1/confusion_matrix.png'
if os.path.exists(cm_img):
    display(Image(cm_img))

## 7. Download Best Checkpoint for Local Inference
Download `best.pt` to save in your local `weights/best.pt` directory for Day 4 (real-time video tracking).

In [ ]:
from google.colab import files
best_weights = 'runs/train/traffic_kitti_v1/weights/best.pt'
if os.path.exists(best_weights):
    files.download(best_weights)
    print('Downloading best.pt to local machine!')
else:
    print(f'Weight file not found at {best_weights}')